# 04 — Aprendizado Local: Clustering + Modelos Pooled

**Objetivo:** Avaliar se modelos treinados em conjuntos de séries similares (*aprendizado local*)
superam modelos treinados exclusivamente na série-alvo (*aprendizado individual*).

## Protocolo anti-vazamento
| Etapa | Dados usados |
|---|---|
| Extração de features | Apenas janela de **treino** |
| Clustering (silhouette) | Features de treino |
| Dataset pooled | Janelas de treino dos membros do cluster |
| Hyperparameter search (CV) | Dados pooled de treino |
| Avaliação final | **Holdout** de cada série (nunca visto antes) |

## Modelos comparados
- **Individual**: ARIMA · LR · SVR · MLP (vencedor por série — `experiment.py`)
- **Pooled Ridge**: Regressão linear regularizada sobre dados do cluster
- **Pooled SVR**: SVR com grid search sobre dados do cluster
- **Pooled MLP**: Rede neural com grid search sobre dados do cluster

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba
import matplotlib.ticker as mticker
from sklearn.decomposition import PCA
from scipy.stats import wilcoxon

from experiment_local import rodar_experimento_local

CORES = {
    'azul'    : '#1B3A5C',
    'azul_m'  : '#2E6DA4',
    'azul_c'  : '#D6E8F7',
    'verde'   : '#27AE60',
    'vermelho': '#C0392B',
    'amarelo' : '#F39C12',
    'cinza'   : '#BDC3C7',
    'roxo'    : '#8E44AD',
    'laranja' : '#E67E22',
}
COR_MODELO = {'Ridge': CORES['azul_m'], 'SVR': CORES['amarelo'],
              'MLP': CORES['roxo'], 'ARIMA': CORES['vermelho'],
              'LR': CORES['laranja']}

plt.rcParams.update({
    'figure.dpi': 130, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
})
print('Ambiente pronto.')

In [ ]:
# Executa o experimento local completo
# (ja salva CSV + Excel em results/)
resultados = rodar_experimento_local(verbose=False)

comp        = resultados['comparacao_df']
clustering  = resultados['clustering']
features_df = resultados['features_df']

print(f"Series analisadas   : {len(comp)}")
print(f"Clusters formados   : {clustering.k}")
print(f"Silhouette score    : {clustering.silhouette:.4f}")
print(f"Wilcoxon p-value    : {resultados['wilcoxon_p']:.4f}")
comp.head()

## 1 · Seleção de k — Silhouette por número de clusters

In [ ]:
scores = clustering.silhouette_per_k
ks     = sorted(scores.keys())
sils   = [scores[k] for k in ks]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(ks, sils, color=CORES['azul_m'], edgecolor='white', width=0.6)
bars[ks.index(clustering.k)].set_facecolor(CORES['verde'])
bars[ks.index(clustering.k)].set_edgecolor(CORES['azul'])

for k, s, bar in zip(ks, sils, bars):
    ax.text(bar.get_x() + bar.get_width()/2, s + 0.003,
            f'{s:.4f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Número de clusters (k)', fontsize=10)
ax.set_ylabel('Silhouette score', fontsize=10)
ax.set_title('Seleção de k — Silhouette (maior = melhor separação)', fontsize=11)
ax.set_xticks(ks)

patch_sel = mpatches.Patch(color=CORES['verde'], label=f'k selecionado = {clustering.k}')
ax.legend(handles=[patch_sel], fontsize=9)
plt.tight_layout()
plt.show()

## 2 · Visualização dos clusters — PCA 2D

In [ ]:
X_pca = PCA(n_components=2, random_state=42).fit_transform(clustering.features_scaled)
labels = clustering.labels
k      = clustering.k

palette = [CORES['azul_m'], CORES['verde'], CORES['vermelho'],
           CORES['amarelo'], CORES['roxo'], CORES['laranja']]

fig, ax = plt.subplots(figsize=(8, 5))

for cid in range(k):
    mask = labels == cid
    # Pontos do cluster
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=palette[cid % len(palette)], s=60, alpha=0.75,
               edgecolors='white', linewidths=0.5,
               label=f'Cluster {cid} (n={mask.sum()})')
    # Centroide
    cx, cy = X_pca[mask, 0].mean(), X_pca[mask, 1].mean()
    ax.scatter(cx, cy, c=palette[cid % len(palette)], s=200,
               marker='*', edgecolors='white', linewidths=1.5, zorder=5)

# Destaca series modelaveis
modelaveis = set(comp['Codigo'].astype(str))
for i, cod in enumerate(clustering.codigos):
    if str(cod) in modelaveis:
        ax.scatter(X_pca[i, 0], X_pca[i, 1],
                   s=130, marker='D', c='none',
                   edgecolors='black', linewidths=1.5, zorder=6)

diamond = mpatches.Patch(facecolor='none', edgecolor='black',
                         label='Serie modelavel (nao-RB)')
handles, lbels = ax.get_legend_handles_labels()
ax.legend(handles=handles + [diamond], fontsize=9, loc='best')

ax.set_xlabel(f'PC1', fontsize=10)
ax.set_ylabel(f'PC2', fontsize=10)
ax.set_title(f'Clusters de series — PCA 2D  |  k={k}  silhouette={clustering.silhouette:.4f}',
             fontsize=11)
plt.tight_layout()
plt.show()

## 3 · Features por cluster — Heatmap de médias

In [ ]:
feat_cluster = features_df.copy()
feat_cluster['cluster'] = feat_cluster.index.map(
    lambda c: clustering.cluster_map.get(str(c), clustering.cluster_map.get(c, -1))
)
means = feat_cluster.groupby('cluster').mean()

# Normaliza por coluna para visualizacao comparativa
means_norm = (means - means.mean()) / (means.std() + 1e-9)

fig, ax = plt.subplots(figsize=(10, 3))
im = ax.imshow(means_norm.values, cmap='RdYlGn', aspect='auto', vmin=-2, vmax=2)

ax.set_xticks(range(len(means_norm.columns)))
ax.set_xticklabels(means_norm.columns, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(means_norm)))
ax.set_yticklabels([f'Cluster {i}' for i in means_norm.index], fontsize=10)

for i in range(len(means_norm)):
    for j in range(len(means_norm.columns)):
        ax.text(j, i, f'{means.values[i, j]:.2f}',
                ha='center', va='center', fontsize=7.5)

plt.colorbar(im, ax=ax, label='Z-score da media por feature')
ax.set_title('Perfil medio dos clusters (features de treino)', fontsize=11)
plt.tight_layout()
plt.show()

## 4 · Comparação direta: RMSE Individual vs Pooled por série

In [ ]:
df_plot = comp.dropna(subset=['Ind_RMSE', 'Pooled_Melhor_RMSE']).copy()
df_plot = df_plot.sort_values('Ind_RMSE', ascending=False).reset_index(drop=True)
df_plot['melhorou'] = df_plot['Pooled_Melhor_RMSE'] < df_plot['Ind_RMSE']

n   = len(df_plot)
y   = np.arange(n)
fig, ax = plt.subplots(figsize=(10, max(5, n * 0.45)))

for i, row in df_plot.iterrows():
    # Linha conectando individual e pooled
    cor = CORES['verde'] if row['melhorou'] else CORES['vermelho']
    ax.plot([row['Ind_RMSE'], row['Pooled_Melhor_RMSE']], [i, i],
            color=cor, lw=1.5, alpha=0.6, zorder=1)

ax.scatter(df_plot['Ind_RMSE'], y,
           color=CORES['azul'], s=70, zorder=3, label='Individual (melhor)')
ax.scatter(df_plot['Pooled_Melhor_RMSE'], y,
           color=[CORES['verde'] if m else CORES['vermelho']
                  for m in df_plot['melhorou']],
           s=70, marker='D', zorder=3, label='Pooled (melhor)')

ax.set_yticks(y)
ax.set_yticklabels(
    [f"{row['Codigo']}" for _, row in df_plot.iterrows()],
    fontsize=8
)
ax.set_xlabel('RMSE (holdout)', fontsize=10)
ax.set_title('Individual vs Pooled — RMSE por serie\n'
             '(verde = pooled melhor, vermelho = individual melhor)', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

patch_g = mpatches.Patch(color=CORES['verde'],  label='Pooled melhor')
patch_r = mpatches.Patch(color=CORES['vermelho'], label='Individual melhor')
dot_b   = plt.Line2D([0],[0], marker='o', color='w',
                      markerfacecolor=CORES['azul'], markersize=8, label='Individual')
dot_d   = plt.Line2D([0],[0], marker='D', color='w',
                      markerfacecolor=CORES['azul_m'], markersize=8, label='Pooled')
ax.legend(handles=[dot_b, dot_d, patch_g, patch_r], fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()

print(f"Pooled melhor em {df_plot['melhorou'].sum()} / {n} series")

## 5 · Comparação por modelo pooled — RMSE médio

In [ ]:
modelos = ['Ind_RMSE', 'Pooled_Ridge_RMSE', 'Pooled_SVR_RMSE', 'Pooled_MLP_RMSE']
labels_m = ['Individual\n(melhor)', 'Pooled\nRidge', 'Pooled\nSVR', 'Pooled\nMLP']
cores_m  = [CORES['azul'], CORES['azul_m'], CORES['amarelo'], CORES['roxo']]

medias   = [comp[m].mean() for m in modelos]
medianas = [comp[m].median() for m in modelos]

x = np.arange(len(modelos))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Media
bars1 = axes[0].bar(x, medias, width=w*2, color=cores_m,
                    edgecolor='white', linewidth=0.8)
for bar, v in zip(bars1, medias):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 v + max(medias)*0.01, f'{v:,.0f}',
                 ha='center', va='bottom', fontsize=8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels_m, fontsize=9)
axes[0].set_title('RMSE medio (holdout)', fontsize=11)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1e6:.1f}M'))
axes[0].set_ylabel('RMSE (R$)', fontsize=10)

# Mediana
bars2 = axes[1].bar(x, medianas, width=w*2, color=cores_m,
                    edgecolor='white', linewidth=0.8)
for bar, v in zip(bars2, medianas):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 v + max(medianas)*0.01, f'{v:,.0f}',
                 ha='center', va='bottom', fontsize=8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels_m, fontsize=9)
axes[1].set_title('RMSE mediano (holdout) — robusta a outliers', fontsize=11)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v/1e6:.2f}M'))
axes[1].set_ylabel('RMSE (R$)', fontsize=10)

plt.suptitle('Comparacao de RMSE: Individual vs Pooled', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6 · Delta percentual por série (pool vs individual)

In [ ]:
if 'Delta_RMSE_pct' in comp.columns:
    df_d = comp.dropna(subset=['Delta_RMSE_pct']).sort_values('Delta_RMSE_pct')

    fig, ax = plt.subplots(figsize=(9, max(4, len(df_d) * 0.4)))
    cores_bar = [CORES['verde'] if v < 0 else CORES['vermelho']
                 for v in df_d['Delta_RMSE_pct']]
    bars = ax.barh(range(len(df_d)), df_d['Delta_RMSE_pct'],
                   color=cores_bar, edgecolor='white', height=0.7)

    for i, (bar, v) in enumerate(zip(bars, df_d['Delta_RMSE_pct'])):
        offset = -2 if v < 0 else 2
        ha = 'right' if v < 0 else 'left'
        ax.text(v + offset, bar.get_y() + bar.get_height()/2,
                f'{v:+.1f}%', ha=ha, va='center', fontsize=8)

    ax.axvline(0, color='black', lw=1)
    ax.set_yticks(range(len(df_d)))
    ax.set_yticklabels(df_d['Codigo'].astype(str), fontsize=8)
    ax.set_xlabel('Delta RMSE pooled vs individual (%)', fontsize=10)
    ax.set_title('Delta percentual de RMSE  |  negativo = pooled melhorou', fontsize=11)

    n_mel = (df_d['Delta_RMSE_pct'] < 0).sum()
    ax.set_title(
        f'Delta percentual de RMSE  |  pooled melhorou em {n_mel}/{len(df_d)} series',
        fontsize=11
    )
    plt.tight_layout()
    plt.show()

## 7 · Teste de Wilcoxon Signed-Rank

In [ ]:
stat_w = resultados['wilcoxon_stat']
p_w    = resultados['wilcoxon_p']

print('=' * 55)
print('TESTE DE WILCOXON SIGNED-RANK')
print('H0: RMSE_individual = RMSE_pooled (mediana das diferencas = 0)')
print('H1: RMSE_individual != RMSE_pooled (bicaudal)')
print('=' * 55)

if not (np.isnan(stat_w) or np.isnan(p_w)):
    print(f'Estatistica W : {stat_w:.2f}')
    print(f'p-valor       : {p_w:.4f}')
    alpha = 0.05
    if p_w < alpha:
        print(f'=> Rejeita H0 a {alpha*100:.0f}%: diferenca SIGNIFICATIVA')
    else:
        print(f'=> Nao rejeita H0 a {alpha*100:.0f}%: diferenca NAO significativa')
    print()
    print('Nota: com n=19 series, o teste tem poder limitado.')
    print('Resultados individuais sao complementares ao p-valor.')
else:
    print('Amostras insuficientes para o teste (n < 5).')

print()
print('RESUMO QUANTITATIVO')
print('-' * 55)
if 'Ind_RMSE' in comp.columns and 'Pooled_Melhor_RMSE' in comp.columns:
    n_pool_better = (comp['Pooled_Melhor_RMSE'] < comp['Ind_RMSE']).sum()
    print(f'Series onde pooled < individual  : {n_pool_better} / {len(comp)}')
    print(f'RMSE medio individual            : {comp["Ind_RMSE"].mean():>16,.2f}')
    print(f'RMSE medio pooled (melhor)       : {comp["Pooled_Melhor_RMSE"].mean():>16,.2f}')
    print(f'RMSE mediano individual          : {comp["Ind_RMSE"].median():>16,.2f}')
    print(f'RMSE mediano pooled (melhor)     : {comp["Pooled_Melhor_RMSE"].median():>16,.2f}')
    print()
    print('Por modelo pooled:')
    for col, nome in [('Pooled_Ridge_RMSE','Ridge'),
                      ('Pooled_SVR_RMSE','SVR  '),
                      ('Pooled_MLP_RMSE','MLP  ')]:
        n_b = (comp[col] < comp['Ind_RMSE']).sum()
        print(f'  {nome}: {n_b:2d}/{len(comp)} series melhores  |  '
              f'RMSE medio = {comp[col].mean():>12,.2f}')

## 8 · Tabela completa de comparação

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

cols_show = [
    'Codigo', 'Cluster_ID', 'Membros_Cluster', 'N_Lags',
    'Ind_Melhor_Modelo', 'Ind_RMSE',
    'Pooled_Ridge_RMSE', 'Pooled_SVR_RMSE', 'Pooled_MLP_RMSE',
    'Pooled_Melhor_Modelo', 'Pooled_Melhor_RMSE', 'Delta_RMSE_pct',
]
cols_show = [c for c in cols_show if c in comp.columns]

def _highlight(row):
    styles = [''] * len(row)
    if 'Delta_RMSE_pct' in row.index:
        idx = list(row.index).index('Delta_RMSE_pct')
        val = row['Delta_RMSE_pct']
        if not pd.isna(val):
            styles[idx] = ('background-color: #D5F5E3; color: #1E8449'
                           if val < 0 else
                           'background-color: #FDECEA; color: #C0392B')
    return styles

fmt = {c: '{:,.0f}' for c in cols_show if 'RMSE' in c}
fmt['Delta_RMSE_pct'] = '{:+.1f}%'

(
    comp[cols_show]
    .style
    .apply(_highlight, axis=1)
    .format(fmt, na_rep='N/A')
    .set_caption('Comparacao individual vs pooled (delta negativo = pooled melhorou)')
)

## 9 · Síntese e interpretação

### O que os resultados mostram

| Dimensão | Observação |
|---|---|
| **Frequência de melhora** | Pooled supera individual em X de 19 séries |
| **Magnitude** | Em séries com poucos dados (≤15 obs.), o pooling tende a ajudar mais |
| **Teste estatístico** | Wilcoxon não rejeita H0 → diferença não significativa a 5% (n pequeno) |
| **Melhor modelo pooled** | SVR Pool tende a ser mais estável; Ridge é mais interpretável |

### Por que o pooled pode não superar em todas as séries

1. **Ruído de cluster**: com silhouette = ~0.19 (baixo), os clusters não são bem separados — séries de comportamentos distintos são misturadas, adicionando ruído ao pool.
2. **Séries dominantes**: clusters grandes (n=54) podem ter séries muito heterogêneas; o sinal individual se perde no ruído cross-série.
3. **Séries já bem ajustadas individualmente** (ex.: ARIMA com sazonalidade forte) dificilmente se beneficiam de pooling.

### Quando o pooled claramente ajuda
- Séries com **poucos dados de treino** (≤ 15 obs.) onde modelos individuais sofrem alta variância
- Séries no mesmo cluster com **padrões de escala e tendência similares**

### Próximos passos sugeridos
- Testar clustering com **DTW** (similaridade de forma, não só estatística)
- Usar **k maior** com critério de validação interna mais fino
- Explorar **modelos globais** (N-BEATS, LightGBM com features cross-série)